# SemCons-IDS — High-Performance Edition v4

**Objectif :** retrouver les performances élevées du pipeline initial sans falsifier les métriques.

Cette édition évalue séparément :

1. **MiniLM-L6-v2 + LoRA** sur une sérialisation compacte des flux ;
2. **XGBoost numérique** sur les caractéristiques réseau ;
3. **fusion tardive** MiniLM–XGBoost apprise sur la partition de calibration ;
4. sélection finale du meilleur candidat sur la calibration, puis évaluation unique sur le test.

> La cible de 99 % concerne généralement le **Weighted-F1** sur un split IID naturel.  
> Le notebook affiche aussi obligatoirement le **Macro-F1**, le MCC et les résultats par classe.

## 0 — Installation compatible Colab

In [ ]:
import os
import sys
import signal
import subprocess
import importlib.metadata as metadata

PINNED = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
    "xgboost": "2.1.4",
}

def version_of(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def stack_ok():
    code = (
        "from transformers import AutoTokenizer, AutoModelForSequenceClassification;"
        "from peft import LoraConfig, TaskType, get_peft_model;"
        "from xgboost import XGBClassifier;"
        "print('STACK_OK')"
    )
    result = subprocess.run(
        [sys.executable, "-c", code],
        text=True,
        capture_output=True,
    )
    return result.returncode == 0 and "STACK_OK" in result.stdout

installed = {name: version_of(name) for name in PINNED}
torchao_version = version_of("torchao")
needs_repair = (
    torchao_version is not None
    or any(installed[name] != expected for name, expected in PINNED.items())
    or not stack_ok()
)

print("Versions avant vérification :")
for name, expected in PINNED.items():
    print(f"  {name:14s}: {installed[name]} (attendu : {expected})")
print(f"  {'torchao':14s}: {torchao_version or 'absent'}")

if needs_repair:
    print("\nRéinstallation propre de la pile...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "uninstall", "-y",
            "transformers", "peft", "accelerate", "tokenizers",
            "torchao", "xgboost",
        ],
        check=False,
    )
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--no-cache-dir",
            "transformers==4.57.6",
            "peft==0.18.0",
            "accelerate==1.12.0",
            "xgboost==2.1.4",
            "pyarrow>=15.0.0",
            "imbalanced-learn>=0.12.0",
            "joblib>=1.3.0",
            "psutil>=5.9.0",
        ]
    )
    if not stack_ok():
        raise RuntimeError("La pile réinstallée ne passe pas le test d'importation.")
    print("✅ Installation réparée.")
    print("♻️ Redémarrage automatique du noyau...")
    os.kill(os.getpid(), signal.SIGKILL)

print("✅ Environnement stable et prêt.")

## 1 — Imports et configuration

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import importlib.metadata as metadata
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

EXPECTED = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
    "xgboost": "2.1.4",
}
for package, expected in EXPECTED.items():
    found = metadata.version(package)
    if found != expected:
        raise RuntimeError(
            f"{package}={found}; attendu={expected}. "
            "Réexécutez la cellule d'installation et laissez le noyau redémarrer."
        )

@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/Dataset/CIC-IDS-2017/"
    output_dir: str = "/content/drive/MyDrive/these_llm/SemCons_IDS_HighPerformance_v4/"
    model_checkpoint: str = "sentence-transformers/all-MiniLM-L6-v2"

    seed: int = 42
    split_mode: str = "iid_stratified"
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    cal_fraction: float = 0.10
    test_fraction: float = 0.10

    min_samples_per_class: int = 200
    numeric_top_k: int = 60
    text_top_k: int = 20
    feature_selection_rows_per_class: int = 20_000

    llm_train_cap_per_class: int = 20_000
    llm_val_cap_per_class: int = 5_000
    max_length: int = 256

    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    epochs: int = 8
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    batch_train: int = 32
    batch_eval: int = 128
    grad_accum: int = 1
    warmup_fraction: float = 0.06
    label_smoothing: float = 0.01
    early_stopping_patience: int = 2

    use_sqrt_class_weights: bool = True
    class_weight_min: float = 0.60
    class_weight_max: float = 2.50

    xgb_n_estimators: int = 1200
    xgb_max_depth: int = 12
    xgb_learning_rate: float = 0.08
    xgb_subsample: float = 0.90
    xgb_colsample_bytree: float = 0.90
    xgb_min_child_weight: float = 1.0
    xgb_reg_lambda: float = 1.0
    xgb_early_stopping_rounds: int = 50

    fusion_selection_metric: str = "weighted_f1"
    fusion_min_semantic_weight: float = 0.00
    fusion_grid_points: int = 41

    target_weighted_f1: float = 0.99

CFG = Config()

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG.seed)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive non monté :", exc)

Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))
print("Device :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GiB")

## 2 — Chargement universel et nettoyage

In [ ]:
def normalize_column_name(name: str) -> str:
    value = str(name).strip().lower()
    value = re.sub(r"[\s/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")

def normalize_label(value: object) -> str | None:
    raw = str(value).strip()
    key = re.sub(r"\s+", " ", raw.lower().replace("_", " ")).strip()

    mapping = {
        "benign": "BENIGN",
        "bot": "Bot",
        "ddos": "DDoS",
        "dos hulk": "DoS_Hulk",
        "dos goldeneye": "DoS_GoldenEye",
        "dos slowhttptest": "DoS_Slowhttptest",
        "dos slowloris": "DoS_Slowloris",
        "ftp-patator": "FTP-Patator",
        "ftp patator": "FTP-Patator",
        "ssh-patator": "SSH-Patator",
        "ssh patator": "SSH-Patator",
        "portscan": "PortScan",
        "port scan": "PortScan",
        "infiltration": "Infiltration",
        "heartbleed": "Heartbleed",
    }
    return mapping.get(key)

def find_data_files(data_dir: str) -> List[Path]:
    root = Path(data_dir)
    files = []
    for pattern in ("*.parquet", "*.csv", "*.feather"):
        files.extend(root.glob(pattern))
    return sorted(files)

def read_one_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        frame = pd.read_parquet(path)
    elif suffix == ".feather":
        frame = pd.read_feather(path)
    elif suffix == ".csv":
        frame = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(path)
    frame.columns = [normalize_column_name(c) for c in frame.columns]
    frame["__source_file__"] = path.name
    return frame

files = find_data_files(CFG.data_dir)
if not files:
    raise FileNotFoundError(f"Aucun CSV/Parquet/Feather dans {CFG.data_dir}")

frames = []
for file_path in files:
    frame = read_one_file(file_path)
    frames.append(frame)
    print(f"{file_path.name:<65s} {len(frame):>10,} lignes")

df = pd.concat(frames, ignore_index=True)
label_candidates = [c for c in df.columns if c == "label" or "label" in c or "class" in c]
if not label_candidates:
    raise ValueError("Colonne label introuvable.")
LABEL_COL = label_candidates[0]

df[LABEL_COL] = df[LABEL_COL].map(normalize_label)
unmapped = int(df[LABEL_COL].isna().sum())
df = df.dropna(subset=[LABEL_COL]).copy()

before_dedup = len(df)
dedup_columns = [c for c in df.columns if c != "__source_file__"]
df = df.drop_duplicates(subset=dedup_columns).reset_index(drop=True)
print(f"\nLabels non reconnus supprimés : {unmapped:,}")
print(f"Doublons supprimés            : {before_dedup - len(df):,}")

numeric_columns = [
    c for c in df.columns
    if c not in {LABEL_COL, "__source_file__"}
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[numeric_columns] = df[numeric_columns].replace([np.inf, -np.inf], np.nan)

counts = df[LABEL_COL].value_counts()
keep_classes = counts[counts >= CFG.min_samples_per_class].index.tolist()
removed_classes = sorted(set(counts.index) - set(keep_classes))
df = df[df[LABEL_COL].isin(keep_classes)].reset_index(drop=True)

print(f"Dataset retenu : {len(df):,} lignes × {len(numeric_columns)} variables candidates")
print("Classes supprimées :", removed_classes)
display(df[LABEL_COL].value_counts().to_frame("effectif"))

## 3 — Split IID stratifié 70/10/10/10

In [ ]:
if CFG.split_mode != "iid_stratified":
    raise ValueError("Cette édition performance utilise explicitement split_mode='iid_stratified'.")

train_df, temp_df = train_test_split(
    df,
    test_size=1.0 - CFG.train_fraction,
    stratify=df[LABEL_COL],
    random_state=CFG.seed,
)
relative_val = CFG.val_fraction / (CFG.val_fraction + CFG.cal_fraction + CFG.test_fraction)
val_df, remain_df = train_test_split(
    temp_df,
    test_size=1.0 - relative_val,
    stratify=temp_df[LABEL_COL],
    random_state=CFG.seed + 1,
)
relative_cal = CFG.cal_fraction / (CFG.cal_fraction + CFG.test_fraction)
cal_df, test_df = train_test_split(
    remain_df,
    test_size=1.0 - relative_cal,
    stratify=remain_df[LABEL_COL],
    random_state=CFG.seed + 2,
)

splits = {
    "train": train_df.reset_index(drop=True),
    "val": val_df.reset_index(drop=True),
    "cal": cal_df.reset_index(drop=True),
    "test": test_df.reset_index(drop=True),
}

for name, split in splits.items():
    print(f"\n{name.upper():5s}: {len(split):,} lignes")
    print(split[LABEL_COL].value_counts().to_string())

## 4 — Prétraitement appris sur le train uniquement

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(splits["train"][LABEL_COL])
class_names = label_encoder.classes_.tolist()
N_CLASSES = len(class_names)

feature_columns = [
    c for c in numeric_columns
    if c in splits["train"].columns
    and c not in {"flow_id", "source_ip", "destination_ip", "timestamp"}
]

X_train_all = splits["train"][feature_columns].to_numpy(dtype=np.float64)
y_train_all = label_encoder.transform(splits["train"][LABEL_COL])

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train_all)

variance_filter = VarianceThreshold(0.0)
X_train_var = variance_filter.fit_transform(X_train_imp)
variance_features = [
    feature for feature, keep in zip(feature_columns, variance_filter.get_support())
    if keep
]

def capped_indices_per_class(y: np.ndarray, cap: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    selected = []
    for cls in np.unique(y):
        indices = np.flatnonzero(y == cls)
        if len(indices) > cap:
            indices = rng.choice(indices, size=cap, replace=False)
        selected.append(indices)
    return np.sort(np.concatenate(selected))

selection_indices = capped_indices_per_class(
    y_train_all,
    CFG.feature_selection_rows_per_class,
    CFG.seed,
)

selector = RandomForestClassifier(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=CFG.seed,
)
selector.fit(X_train_var[selection_indices], y_train_all[selection_indices])

importance_df = pd.DataFrame({
    "feature": variance_features,
    "importance": selector.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

NUMERIC_FEATURES = importance_df.head(min(CFG.numeric_top_k, len(importance_df)))["feature"].tolist()
TEXT_FEATURES = NUMERIC_FEATURES[:min(CFG.text_top_k, len(NUMERIC_FEATURES))]

variance_index = {name: idx for idx, name in enumerate(variance_features)}
numeric_positions = [variance_index[name] for name in NUMERIC_FEATURES]
text_positions = [NUMERIC_FEATURES.index(name) for name in TEXT_FEATURES]

scaler = StandardScaler()

arrays: Dict[str, Dict[str, np.ndarray]] = {}
for split_name, split_df in splits.items():
    X = split_df[feature_columns].to_numpy(dtype=np.float64)
    X_imp = imputer.transform(X)
    X_var = variance_filter.transform(X_imp)
    X_num = X_var[:, numeric_positions].astype(np.float32)
    if split_name == "train":
        X_scaled = scaler.fit_transform(X_num).astype(np.float32)
    else:
        X_scaled = scaler.transform(X_num).astype(np.float32)

    arrays[split_name] = {
        "raw_numeric": X_num,
        "scaled_numeric": X_scaled,
        "scaled_text": X_scaled[:, text_positions],
        "y": label_encoder.transform(split_df[LABEL_COL]).astype(np.int64),
    }

print("Classes :", class_names)
print("\nTop caractéristiques numériques :")
display(importance_df.head(len(NUMERIC_FEATURES)))
print("\nCaractéristiques textuelles :", TEXT_FEATURES)

preprocess_dir = Path(CFG.output_dir) / "preprocessing"
preprocess_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(label_encoder, preprocess_dir / "label_encoder.joblib")
joblib.dump(imputer, preprocess_dir / "imputer.joblib")
joblib.dump(variance_filter, preprocess_dir / "variance_filter.joblib")
joblib.dump(scaler, preprocess_dir / "scaler.joblib")
joblib.dump(feature_columns, preprocess_dir / "input_features.joblib")
joblib.dump(NUMERIC_FEATURES, preprocess_dir / "numeric_features.joblib")
joblib.dump(TEXT_FEATURES, preprocess_dir / "text_features.joblib")

## 5 — Sous-ensembles MiniLM et poids de classes atténués

In [ ]:
def cap_subset_indices(y: np.ndarray, cap: int | None, seed: int) -> np.ndarray:
    if cap is None:
        return np.arange(len(y))
    return capped_indices_per_class(y, cap, seed)

llm_train_indices = cap_subset_indices(
    arrays["train"]["y"], CFG.llm_train_cap_per_class, CFG.seed + 10
)
llm_val_indices = cap_subset_indices(
    arrays["val"]["y"], CFG.llm_val_cap_per_class, CFG.seed + 11
)

train_counts = np.bincount(
    arrays["train"]["y"][llm_train_indices],
    minlength=N_CLASSES,
).astype(np.float64)

if CFG.use_sqrt_class_weights:
    weights = np.sqrt(
        len(llm_train_indices) / (N_CLASSES * np.maximum(train_counts, 1.0))
    )
    weights = np.clip(weights, CFG.class_weight_min, CFG.class_weight_max)
    weights = weights / weights.mean()
else:
    weights = np.ones(N_CLASSES, dtype=np.float64)

CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32)

print("MiniLM train :", len(llm_train_indices))
print("MiniLM val   :", len(llm_val_indices))
print("Poids :", {
    class_names[i]: round(float(weights[i]), 4)
    for i in range(N_CLASSES)
})

## 6 — Sérialisation compacte et Dataset MiniLM

In [ ]:
def compact_prompt(values: np.ndarray) -> str:
    return "classify: " + " ".join(
        f"{name}={float(value):.4f}"
        for name, value in zip(TEXT_FEATURES, values)
    )

tokenizer = AutoTokenizer.from_pretrained(CFG.model_checkpoint)

sample_prompt = compact_prompt(arrays["train"]["scaled_text"][llm_train_indices[0]])
sample_tokens = tokenizer(
    sample_prompt,
    truncation=False,
    add_special_tokens=True,
)["input_ids"]

print("Exemple :", sample_prompt[:500])
print("Tokens  :", len(sample_tokens))

audit_n = min(10_000, len(llm_train_indices))
audit_indices = llm_train_indices[:audit_n]
audit_lengths = [
    len(tokenizer(
        compact_prompt(arrays["train"]["scaled_text"][idx]),
        truncation=False,
        add_special_tokens=True,
    )["input_ids"])
    for idx in tqdm(audit_indices, desc="Audit tokens")
]
print("Longueur moyenne :", round(float(np.mean(audit_lengths)), 2))
print("P95              :", round(float(np.percentile(audit_lengths, 95)), 2))
print("Taux tronqué     :", round(float(np.mean(np.asarray(audit_lengths) > CFG.max_length)), 6))

class IndexedDataset(Dataset):
    def __init__(self, labels: np.ndarray, indices: np.ndarray):
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = int(self.indices[item])
        return {"index": idx, "label": int(self.labels[idx])}

class CompactCollator:
    def __init__(
        self,
        tokenizer,
        scaled_text: np.ndarray,
        max_length: int,
    ):
        self.tokenizer = tokenizer
        self.scaled_text = scaled_text
        self.max_length = max_length

    def __call__(self, items):
        indices = [item["index"] for item in items]
        labels = torch.tensor([item["label"] for item in items], dtype=torch.long)
        texts = [compact_prompt(self.scaled_text[idx]) for idx in indices]
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        encoded["labels"] = labels
        encoded["indices"] = torch.tensor(indices, dtype=torch.long)
        return encoded

train_dataset = IndexedDataset(arrays["train"]["y"], llm_train_indices)
val_dataset = IndexedDataset(arrays["val"]["y"], llm_val_indices)

train_collator = CompactCollator(
    tokenizer, arrays["train"]["scaled_text"], CFG.max_length
)
val_collator = CompactCollator(
    tokenizer, arrays["val"]["scaled_text"], CFG.max_length
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_train,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=train_collator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.batch_eval,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=val_collator,
)

batch = next(iter(train_loader))
print({key: tuple(value.shape) for key, value in batch.items()})

## 7 — MiniLM + LoRA

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    CFG.model_checkpoint,
    num_labels=N_CLASSES,
    id2label={i: name for i, name in enumerate(class_names)},
    label2id={name: i for i, name in enumerate(class_names)},
    ignore_mismatched_sizes=True,
)

linear_leaf_names = sorted({
    name.split(".")[-1]
    for name, module in base_model.named_modules()
    if isinstance(module, nn.Linear)
})
print("Modules linéaires :", linear_leaf_names)

targets = ["query", "value"]
missing = [name for name in targets if name not in linear_leaf_names]
if missing:
    raise ValueError(f"Modules LoRA introuvables : {missing}")

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    target_modules=targets,
    modules_to_save=["classifier"],
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(base_model, lora_config).to(DEVICE)

n_total = sum(parameter.numel() for parameter in model.parameters())
n_trainable = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
n_lora = sum(
    parameter.numel()
    for name, parameter in model.named_parameters()
    if parameter.requires_grad and "lora_" in name
)
n_head = n_trainable - n_lora

print(f"Total             : {n_total:,}")
print(f"LoRA entraînable  : {n_lora:,} ({100*n_lora/n_total:.4f}%)")
print(f"Tête entraînable  : {n_head:,} ({100*n_head/n_total:.4f}%)")
print(f"Total entraînable : {n_trainable:,} ({100*n_trainable/n_total:.4f}%)")

## 8 — Métriques et entraînement MiniLM

In [ ]:
def softmax_np(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)

def metrics_from_probs(
    probabilities: np.ndarray,
    labels: np.ndarray,
) -> Dict[str, float]:
    predictions = probabilities.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "mcc": float(matthews_corrcoef(labels, predictions)),
        "macro_precision": float(
            precision_score(labels, predictions, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(labels, predictions, average="macro", zero_division=0)
        ),
        "nll": float(log_loss(labels, probabilities, labels=np.arange(N_CLASSES))),
    }

@torch.no_grad()
def collect_llm_probs(loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_probs, all_labels, all_indices = [], [], []
    for batch in tqdm(loader, leave=False):
        labels = batch.pop("labels")
        indices = batch.pop("indices")
        batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
        logits = model(**batch).logits
        all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
        all_labels.append(labels.numpy())
        all_indices.append(indices.numpy())
    return (
        np.concatenate(all_probs),
        np.concatenate(all_labels),
        np.concatenate(all_indices),
    )

criterion = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS.to(DEVICE),
    label_smoothing=CFG.label_smoothing,
)
optimizer = AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)

steps_per_epoch = math.ceil(len(train_loader) / CFG.grad_accum)
total_steps = max(1, CFG.epochs * steps_per_epoch)
warmup_steps = int(CFG.warmup_fraction * total_steps)

def schedule_lambda(step: int) -> float:
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda=schedule_lambda)
amp_enabled = DEVICE.type == "cuda"
grad_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

best_state = None
best_val_macro = -np.inf
best_epoch = -1
patience = 0
history = []

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
train_start = time.perf_counter()

for epoch in range(1, CFG.epochs + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0

    progress = tqdm(train_loader, desc=f"MiniLM epoch {epoch}/{CFG.epochs}")
    for step, batch in enumerate(progress, start=1):
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        batch.pop("indices")
        batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            logits = model(**batch).logits
            loss = criterion(logits, labels) / CFG.grad_accum

        grad_scaler.scale(loss).backward()
        running_loss += float(loss.detach().cpu()) * CFG.grad_accum

        if step % CFG.grad_accum == 0 or step == len(train_loader):
            grad_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [parameter for parameter in model.parameters() if parameter.requires_grad],
                max_norm=1.0,
            )
            grad_scaler.step(optimizer)
            grad_scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        progress.set_postfix(loss=f"{running_loss/step:.4f}")

    val_probs, val_labels_small, _ = collect_llm_probs(val_loader)
    val_metrics = metrics_from_probs(val_probs, val_labels_small)
    row = {
        "epoch": epoch,
        "train_loss": running_loss / max(1, len(train_loader)),
        **{f"val_{key}": value for key, value in val_metrics.items()},
    }
    history.append(row)
    print(pd.Series(row).round(6))

    if val_metrics["macro_f1"] > best_val_macro:
        best_val_macro = val_metrics["macro_f1"]
        best_epoch = epoch
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        patience = 0
        print(f"🏆 Nouveau meilleur MiniLM : Macro-F1={best_val_macro:.6f}")
    else:
        patience += 1
        if patience >= CFG.early_stopping_patience:
            print("Early stopping MiniLM.")
            break

if best_state is None:
    raise RuntimeError("Aucun checkpoint MiniLM sauvegardé.")

model.load_state_dict(best_state, strict=True)
model.eval()

llm_train_seconds = time.perf_counter() - train_start
peak_vram_gib = (
    torch.cuda.max_memory_allocated() / 2**30
    if DEVICE.type == "cuda"
    else 0.0
)
history_df = pd.DataFrame(history)
display(history_df)
print(f"Best epoch : {best_epoch}")
print(f"Temps      : {llm_train_seconds/60:.2f} min")
print(f"VRAM peak  : {peak_vram_gib:.3f} GiB")

## 9 — Expert numérique XGBoost

In [ ]:
xgb_common = dict(
    objective="multi:softprob",
    num_class=N_CLASSES,
    n_estimators=CFG.xgb_n_estimators,
    max_depth=CFG.xgb_max_depth,
    learning_rate=CFG.xgb_learning_rate,
    subsample=CFG.xgb_subsample,
    colsample_bytree=CFG.xgb_colsample_bytree,
    min_child_weight=CFG.xgb_min_child_weight,
    reg_lambda=CFG.xgb_reg_lambda,
    max_bin=256,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=CFG.xgb_early_stopping_rounds,
    random_state=CFG.seed,
    n_jobs=-1,
)

xgb_model = XGBClassifier(
    **xgb_common,
    device="cuda" if DEVICE.type == "cuda" else "cpu",
)

xgb_start = time.perf_counter()
try:
    xgb_model.fit(
        arrays["train"]["raw_numeric"],
        arrays["train"]["y"],
        eval_set=[(arrays["val"]["raw_numeric"], arrays["val"]["y"])],
        verbose=50,
    )
except Exception as exc:
    print("⚠️ XGBoost GPU indisponible, reprise CPU :", exc)
    xgb_model = XGBClassifier(**xgb_common, device="cpu")
    xgb_model.fit(
        arrays["train"]["raw_numeric"],
        arrays["train"]["y"],
        eval_set=[(arrays["val"]["raw_numeric"], arrays["val"]["y"])],
        verbose=50,
    )

xgb_train_seconds = time.perf_counter() - xgb_start
print(f"Temps XGBoost : {xgb_train_seconds/60:.2f} min")
print("Best iteration :", getattr(xgb_model, "best_iteration", None))

## 10 — Probabilités MiniLM et XGBoost sur validation/calibration/test

In [ ]:
def make_full_llm_loader(split_name: str) -> DataLoader:
    indices = np.arange(len(arrays[split_name]["y"]))
    dataset = IndexedDataset(arrays[split_name]["y"], indices)
    collator = CompactCollator(
        tokenizer,
        arrays[split_name]["scaled_text"],
        CFG.max_length,
    )
    return DataLoader(
        dataset,
        batch_size=CFG.batch_eval,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collator,
    )

probabilities: Dict[str, Dict[str, np.ndarray]] = {}

for split_name in ["val", "cal", "test"]:
    print(f"\nInférence {split_name.upper()}...")
    llm_loader = make_full_llm_loader(split_name)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    llm_probs, labels, indices = collect_llm_probs(llm_loader)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    llm_seconds = time.perf_counter() - start

    xgb_probs = xgb_model.predict_proba(arrays[split_name]["raw_numeric"])
    xgb_probs = np.asarray(xgb_probs, dtype=np.float64)
    xgb_probs = xgb_probs / xgb_probs.sum(axis=1, keepdims=True)

    probabilities[split_name] = {
        "llm": llm_probs.astype(np.float64),
        "xgb": xgb_probs,
        "labels": labels,
        "indices": indices,
        "llm_seconds": np.asarray([llm_seconds]),
    }

    print("MiniLM :", pd.Series(metrics_from_probs(llm_probs, labels)).round(6).to_dict())
    print("XGBoost:", pd.Series(metrics_from_probs(xgb_probs, labels)).round(6).to_dict())

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 11 — Fusion tardive et sélection sur la calibration

In [ ]:
def selection_score(metrics: Dict[str, float]) -> float:
    if CFG.fusion_selection_metric not in metrics:
        raise KeyError(CFG.fusion_selection_metric)
    return metrics[CFG.fusion_selection_metric]

cal_labels = probabilities["cal"]["labels"]
cal_llm = probabilities["cal"]["llm"]
cal_xgb = probabilities["cal"]["xgb"]

candidate_cal_metrics: Dict[str, Dict[str, float]] = {}
candidate_test_probs: Dict[str, np.ndarray] = {}

candidate_cal_metrics["minilm"] = metrics_from_probs(cal_llm, cal_labels)
candidate_test_probs["minilm"] = probabilities["test"]["llm"]

candidate_cal_metrics["xgboost"] = metrics_from_probs(cal_xgb, cal_labels)
candidate_test_probs["xgboost"] = probabilities["test"]["xgb"]

# Fusion convexe : alpha = poids numérique, (1-alpha) = poids MiniLM.
minimum_alpha = 0.0
maximum_alpha = 1.0 - CFG.fusion_min_semantic_weight
alphas = np.linspace(minimum_alpha, maximum_alpha, CFG.fusion_grid_points)

best_alpha = None
best_alpha_metrics = None
best_alpha_score = -np.inf

for alpha in alphas:
    fused = alpha * cal_xgb + (1.0 - alpha) * cal_llm
    fused = fused / fused.sum(axis=1, keepdims=True)
    metrics = metrics_from_probs(fused, cal_labels)
    score = selection_score(metrics)
    if score > best_alpha_score:
        best_alpha_score = score
        best_alpha = float(alpha)
        best_alpha_metrics = metrics

candidate_cal_metrics[f"fusion_alpha_{best_alpha:.3f}"] = best_alpha_metrics
test_alpha_probs = (
    best_alpha * probabilities["test"]["xgb"]
    + (1.0 - best_alpha) * probabilities["test"]["llm"]
)
test_alpha_probs /= test_alpha_probs.sum(axis=1, keepdims=True)
candidate_test_probs[f"fusion_alpha_{best_alpha:.3f}"] = test_alpha_probs

# Stacking multinomial sur log-probabilités.
eps = 1e-8
stack_X_cal = np.concatenate(
    [np.log(cal_xgb + eps), np.log(cal_llm + eps)],
    axis=1,
)
stacker = LogisticRegression(
    C=10.0,
    max_iter=1000,
    solver="lbfgs",
    multi_class="multinomial",
    n_jobs=-1,
    random_state=CFG.seed,
)
stacker.fit(stack_X_cal, cal_labels)

stack_cal_probs = stacker.predict_proba(stack_X_cal)
candidate_cal_metrics["stacking"] = metrics_from_probs(stack_cal_probs, cal_labels)

stack_X_test = np.concatenate(
    [
        np.log(probabilities["test"]["xgb"] + eps),
        np.log(probabilities["test"]["llm"] + eps),
    ],
    axis=1,
)
candidate_test_probs["stacking"] = stacker.predict_proba(stack_X_test)

comparison_cal = pd.DataFrame(candidate_cal_metrics).T.sort_values(
    CFG.fusion_selection_metric,
    ascending=False,
)
display(comparison_cal)

SELECTED_CANDIDATE = comparison_cal.index[0]
print("Candidat sélectionné sur CAL :", SELECTED_CANDIDATE)
print("Alpha numérique optimal       :", best_alpha)

## 12 — Évaluation finale unique sur le test

In [ ]:
test_labels = probabilities["test"]["labels"]
test_results = {}

for candidate_name, candidate_probs in candidate_test_probs.items():
    test_results[candidate_name] = metrics_from_probs(candidate_probs, test_labels)

test_comparison = pd.DataFrame(test_results).T.sort_values(
    CFG.fusion_selection_metric,
    ascending=False,
)
display(test_comparison)

final_probs = candidate_test_probs[SELECTED_CANDIDATE]
final_predictions = final_probs.argmax(axis=1)
final_metrics = test_results[SELECTED_CANDIDATE]

print("\n" + "=" * 76)
print("RÉSULTAT FINAL — SÉLECTION EFFECTUÉE SUR CALIBRATION")
print("=" * 76)
print("Candidat     :", SELECTED_CANDIDATE)
print(f"Weighted-F1  : {final_metrics['weighted_f1']:.6f}")
print(f"Macro-F1     : {final_metrics['macro_f1']:.6f}")
print(f"MCC          : {final_metrics['mcc']:.6f}")
print(f"Accuracy     : {final_metrics['accuracy']:.6f}")
print(f"Balanced Acc : {final_metrics['balanced_accuracy']:.6f}")
print("=" * 76)

if final_metrics["weighted_f1"] >= CFG.target_weighted_f1:
    print(f"✅ Cible atteinte : Weighted-F1 ≥ {CFG.target_weighted_f1:.2%}")
else:
    print(
        f"⚠️ Cible non atteinte : {final_metrics['weighted_f1']:.2%} "
        f"< {CFG.target_weighted_f1:.2%}. Le notebook ne fabrique pas le résultat."
    )

print("\nRapport de classification :")
print(classification_report(
    test_labels,
    final_predictions,
    target_names=class_names,
    digits=4,
    zero_division=0,
))

cm = confusion_matrix(test_labels, final_predictions, labels=np.arange(N_CLASSES))
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
display(cm_df)

## 13 — Analyse de la contribution de chaque branche

In [ ]:
branch_summary = pd.DataFrame({
    "MiniLM-LoRA": test_results["minilm"],
    "XGBoost": test_results["xgboost"],
    "Final": test_results[SELECTED_CANDIDATE],
}).T

branch_summary["delta_weighted_f1_vs_minilm"] = (
    branch_summary["weighted_f1"] - branch_summary.loc["MiniLM-LoRA", "weighted_f1"]
)
branch_summary["delta_macro_f1_vs_minilm"] = (
    branch_summary["macro_f1"] - branch_summary.loc["MiniLM-LoRA", "macro_f1"]
)
display(branch_summary)

importance = pd.DataFrame({
    "feature": NUMERIC_FEATURES,
    "xgb_importance": xgb_model.feature_importances_,
}).sort_values("xgb_importance", ascending=False)
display(importance.head(25))

## 14 — Sauvegarde complète

In [ ]:
output_dir = Path(CFG.output_dir)
adapter_dir = output_dir / "minilm_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
torch.save(best_state, output_dir / "minilm_best_full_state.pt")

joblib.dump(xgb_model, output_dir / "xgboost_model.joblib")
joblib.dump(stacker, output_dir / "fusion_stacker.joblib")

with open(output_dir / "config.json", "w", encoding="utf-8") as handle:
    json.dump(asdict(CFG), handle, indent=2, ensure_ascii=False)

fusion_metadata = {
    "selected_candidate": SELECTED_CANDIDATE,
    "best_alpha_numeric": best_alpha,
    "selection_metric": CFG.fusion_selection_metric,
    "class_names": class_names,
}
with open(output_dir / "fusion_metadata.json", "w", encoding="utf-8") as handle:
    json.dump(fusion_metadata, handle, indent=2, ensure_ascii=False)

history_df.to_csv(output_dir / "minilm_training_history.csv", index=False)
importance_df.to_csv(output_dir / "feature_selection_importance.csv", index=False)
importance.to_csv(output_dir / "xgboost_feature_importance.csv", index=False)
comparison_cal.to_csv(output_dir / "calibration_candidate_comparison.csv")
test_comparison.to_csv(output_dir / "test_candidate_comparison.csv")
branch_summary.to_csv(output_dir / "branch_summary.csv")

prediction_frame = pd.DataFrame({
    "true_label": [class_names[index] for index in test_labels],
    "predicted_label": [class_names[index] for index in final_predictions],
    "confidence": final_probs.max(axis=1),
})
for class_index, class_name in enumerate(class_names):
    prediction_frame[f"p_{class_name}"] = final_probs[:, class_index]
prediction_frame.to_csv(output_dir / "test_predictions.csv", index=False)

summary = {
    "dataset": "CICIDS2017",
    "protocol": CFG.split_mode,
    "selected_candidate": SELECTED_CANDIDATE,
    "best_alpha_numeric": best_alpha,
    "n_classes": N_CLASSES,
    "classes": " | ".join(class_names),
    "numeric_features": " | ".join(NUMERIC_FEATURES),
    "text_features": " | ".join(TEXT_FEATURES),
    "llm_total_parameters": n_total,
    "llm_trainable_parameters": n_trainable,
    "llm_trainable_percent": 100.0 * n_trainable / n_total,
    "llm_train_seconds": llm_train_seconds,
    "xgb_train_seconds": xgb_train_seconds,
    "peak_vram_gib": peak_vram_gib,
    **final_metrics,
}
pd.DataFrame([summary]).to_csv(output_dir / "high_performance_summary.csv", index=False)

print("Pipeline terminé.")
print("Sorties :", output_dir)
print(pd.Series(summary))

## Interprétation scientifique

- **Weighted-F1 ≥ 99 %** peut être favorisé par la forte proportion de trafic `BENIGN`.
- **Macro-F1** reste la mesure principale pour vérifier que les classes rares ne sont pas masquées.
- Le split par lignes IID reproduit les conditions du pipeline de départ, mais il est moins exigeant qu’un split temporel ou inter-fichiers.
- Les résultats de MiniLM, XGBoost et de la fusion sont toujours publiés séparément.